### Eisbudendilemma – Lösung Schritt für Schritt

*39. Bundeswettbewerb Informatik (2020/2021), 2. Runde, Aufgabe 3*

Ein bayrisches Bergdorf besteht aus einer überschaubaren Menge von Häusern, die um den nahezu kreisförmigen Kondorsee – nur durch einen
kleinen Weg von diesem getrennt – liegen. Der Umfang des Sees beträgt 247 Schritte, und Adressen entlang der Straße werden traditionell
in Anzahl von Schritten im Gegenuhrzeigersinn vom Standort der Dorfkirche (sie hat Adresse Null) gezählt. Die Adressen der Häuser sind
2, 5, 37, 43, 72, 74, 83, 87, 93, 97, 101, 110, 121, 124, 126, 136, 150, 161, 185, 200, 201, 230, 234 und 241.

Im Dorf wird hart gearbeitet, und das einzige Vergnügen war schon immer eine Eisspezialität, zu deren Herstellung seit jeher Winterschnee
in Höhlen zu späterem sommerlichen Verzehr gelagert worden war. Dieses Eis wird in drei Eisbuden angeboten, welche sich am Seeufer im
Inneren des Uferwegs befinden und deren Standorte sich seit Ewigkeiten nicht verändert haben.

Vor kurzem hatte eine Bewohnerin im Dorfarchiv ein altes Gesetz entdeckt, welches sich als auch heute noch gültig herausstellte und dessen
Aufhebung ob des konservativen Gemeinderats unmöglich wäre. Dieses Gesetz besagt sinngemäß: Schlägt ein Dorfbewohner neue Standorte für
die Eisbuden vor, dann muß darüber in der wöchentlichen Gemeindesitzung abgestimmt werden, in der jedes Haus mit einer Stimme vertreten
ist. Erhält der neue Standortvorschlag mehr Ja- als Neinstimmen, werden die Eisbuden unverzüglich entsprechend verschoben.
Die Dorfbewohner sind alle Egoisten. Jeder stimmt genau dann für einen neuen Standortvorschlag, wenn es seinen Weg zur nächstgelegenen
Bude verkürzt, ansonsten dagegen.
Nach einigen Monaten sind die Dorfbewohner der ständigen Eisbudenverlegungen müde und wünschten sich, das Gesetz wäre nie entdeckt
worden. Kannst du ihnen helfen und Standorte für die drei Eisbuden finden, die zusammen in dem Sinne stabil sind, dass sie nie wieder
verändert werden können? Es darf also keine anderen Standorte geben, die bei einer Abstimmung eine Mehrheit bekämen.

#### Aufgabe

Schreibe ein Programm, das für einen eingegebenen Teichumfang und eingegebene Adressen von Häusern stabile Eisbudenstandorte findet,
falls solche existieren.

Dein Programm sollte in der Praxis möglichst schnell sein und zumindest für die auf den BWINF-Webseiten gegebenen Beispiele stabile
Standorte berechnen oder feststellen können, dass es solche nicht gibt.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `eisbuden1.txt` bis `eisbuden7.txt`. Die erste Zeile enthält den Umfang `L` und die Zahl der Häuser
`N`, die zweite die Adressen der Häuser (die Dateien haben keine Formatbeschreibung; so lesen wir sie).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Alles durchprobieren (Brute Force):** Es gibt nur `L · (L-1) · (L-2) / 6` Verteilungen von drei Buden. Für jede prüft man, ob sie
  stabil ist. Das ist die Grundlage; danach geht es nur noch darum, schneller zu werden.
- **Kreis und Abstand:** Auf einem Kreis mit Umfang `L` ist der Abstand zweier Adressen `min(|a - b|, L - |a - b|)`.
- **Mengen als Bitmasken:** Die Menge der Häuser, die für eine Adresse stimmen, ist eine ganze Zahl, bei der Bit `h` für Haus `h`
  steht. Die Vereinigung ist dann `m1 | m2`, die Größe zählt man mit `.bit_count()`.
- **Abschätzen und frühzeitig abbrechen:** Sind die Mengen nach Größe sortiert, kann man eine Suche abbrechen, sobald auch die größten
  noch möglichen Mengen zusammen nicht mehr reichen.
- **Gegenbeispiele merken:** Ein Gegenvorschlag W, der eine Verteilung V schlägt, schlägt oft auch viele andere Verteilungen. Wer sich
  solche W merkt, erledigt die meisten Verteilungen mit einem einzigen Vergleich.
- **Lokale Suche (Abstieg):** Man verschiebt eine Bude nach der anderen an die Stelle mit der kleinsten Gesamtentfernung, bis sich
  nichts mehr verbessert.

**Nützliche Funktionen und Bibliotheken**
- `itertools.combinations(range(L), 3)` liefert alle Verteilungen `i < j < k`.
- `int.bit_count()` zählt die gesetzten Bits.
- `numpy` für Tabellen von Abständen und für das Vergleichen sehr vieler Verteilungen auf einmal (nur ab Schritt 6).

## Die Idee

**Abstimmung.** Haus `h` hat zur nächsten Bude den Abstand `d(h)`. Eine neue Verteilung `W` bekommt die Stimme von `h`, wenn **eine** ihrer
Buden näher als `d(h)` an `h` liegt. Das Haus stimmt also für eine **Adresse** `a`, wenn `a` näher an `h` liegt als `d(h)`. Und `W` gewinnt,
wenn mindestens `N // 2 + 1` Häuser für mindestens eine der drei Adressen von `W` stimmen (mehr Ja als Nein, jedes Haus stimmt mit Ja oder
Nein). Eine Verteilung `V` ist **stabil**, wenn kein `W` gewinnt.

**Gebote.** Für feste `V` berechnen wir für jede Adresse `a` die Menge der Häuser, die für `a` stimmen (ihr „Gebot“). `V` ist
instabil genau dann, wenn es **drei** Adressen gibt, deren Mengen zusammen mindestens `N // 2 + 1` Häuser enthalten. Die Mengen
müssen **vereinigt** werden: Ein Haus, das für zwei der drei Adressen stimmt, zählt nur einmal. Das ist das „Gebote-System“ der
Lösungshinweise.

**Alle Verteilungen.** Wir prüfen jede Verteilung `i < j < k` (Siedlungs-Iteration). Das ist für kleine Beispiele schnell genug und
für die großen viel zu langsam. Darum kommen danach zwei Beschleunigungen: Gegenvorschläge merken und mit ihnen fast alle Verteilungen
mit einem einzigen Vergleich erledigen, und zuletzt dasselbe mit `numpy`.

## Schritt 1: Einlesen und Abstimmung

`abstand(a, b, L)` ist der kürzeste Weg auf dem Kreis. `ja_stimmen(V, W, L, haeuser)` zählt, wie viele Häuser für `W` gegen `V`
stimmen. Zur Probe der Beispielkreis der Hinweise: Umfang 12, Häuser an 0, 4 und 8. Die Verteilungen `(0, 1, 2)`, `(4, 5, 6)` und
`(8, 9, 10)` lösen sich reihum ab, jede bekommt 2 von 3 Stimmen. Man kann also nicht einfach so lange verbessern, bis nichts mehr geht.

In [ ]:
def lies(nr):
    with open('beispieldaten/eisbuden%d.txt' % nr) as f:
        zahlen = [int(x) for x in f.read().split()]
    L, N = zahlen[0], zahlen[1]
    return L, zahlen[2:2 + N]


def abstand(a, b, L):
    d = abs(a - b)
    return min(d, L - d)


def ja_stimmen(V, W, L, haeuser):
    anzahl = 0
    for h in haeuser:
        alt = min(abstand(h, b, L) for b in V)
        neu = min(abstand(h, b, L) for b in W)
        if neu < alt:
            anzahl += 1
    return anzahl


kreis = [0, 4, 8]
for V, W in [((0, 1, 2), (4, 5, 6)), ((4, 5, 6), (8, 9, 10)), ((8, 9, 10), (0, 1, 2))]:
    print(V, '->', W, ':', ja_stimmen(V, W, 12, kreis), 'von', len(kreis), 'Stimmen')

(0, 1, 2) -> (4, 5, 6) : 2 von 3 Stimmen
(4, 5, 6) -> (8, 9, 10) : 2 von 3 Stimmen
(8, 9, 10) -> (0, 1, 2) : 2 von 3 Stimmen


## Schritt 2: Gebote und Stabilität einer Verteilung

`gebote(V, L, haeuser)` gibt für jede Adresse mit mindestens einer Stimme ein Tripel `(Anzahl, Adresse, Maske)` zurück, absteigend nach
der Anzahl sortiert. `Maske` hat Bit `h` gesetzt, wenn das Haus mit dem Index `h` für die Adresse stimmt.

`gegenvorschlag(V, L, haeuser)` sucht drei Gebote, die zusammen mindestens `N // 2 + 1` Häuser abdecken, und gibt sie als
`(a, b, c)` zurück, sonst `None`. Weil die Gebote sortiert sind, brechen wir ab, sobald selbst die größten noch möglichen nicht mehr
reichen (Abschätzung nach oben mit den noch nicht ausgewählten Zahlen). Zur Probe das Beispiel der Hinweise: Umfang 17, Häuser an
0, 3, 6, 7, 8, 11, 13, 14, 15 und die Verteilung `(7, 12, 15)`; die Stimmen je Adresse stehen dort in Tabelle 3.

In [ ]:
def gebote(V, L, haeuser):
    nah = [min(abstand(h, b, L) for b in V) for h in haeuser]
    liste = []
    for a in range(L):
        maske = 0
        for i, h in enumerate(haeuser):
            if abstand(h, a, L) < nah[i]:
                maske |= 1 << i
        if maske:
            liste.append((maske.bit_count(), a, maske))
    liste.sort(reverse=True)
    return liste


def gegenvorschlag(V, L, haeuser):
    mehrheit = len(haeuser) // 2 + 1
    g = gebote(V, L, haeuser)
    n = len(g)
    for x in range(n):
        sx, ax, mx = g[x]
        if 3 * sx < mehrheit:             # die weiteren Gebote haben höchstens genauso viele Stimmen
            break
        for y in range(x + 1, n):
            sy, ay, my = g[y]
            if sx + 2 * sy < mehrheit:
                break
            mxy = mx | my
            for z in range(y + 1, n):
                sz, az, mz = g[z]
                if sx + sy + sz < mehrheit:
                    break
                if (mxy | mz).bit_count() >= mehrheit:
                    return (ax, ay, az)
    return None


def ist_stabil(V, L, haeuser):
    return gegenvorschlag(V, L, haeuser) is None


LB = 17
HB = [0, 3, 6, 7, 8, 11, 13, 14, 15]
print('Stimmen je Adresse:', {a: s for s, a, m in sorted(gebote((7, 12, 15), LB, HB), key=lambda t: t[1])})
print('Gegenvorschlag zu (7, 12, 15):', gegenvorschlag((7, 12, 15), LB, HB))
W = gegenvorschlag((0, 1, 2), LB, HB)
print('Gegenvorschlag zu (0, 1, 2):', W, 'mit', ja_stimmen((0, 1, 2), W, LB, HB), 'von', len(HB), 'Stimmen')

Stimmen je Adresse: {0: 2, 1: 2, 2: 1, 3: 1, 4: 1, 5: 1, 6: 2, 8: 1, 11: 1, 13: 1, 14: 1, 16: 1}
Gegenvorschlag zu (7, 12, 15): None
Gegenvorschlag zu (0, 1, 2): (16, 15, 13) mit 5 von 9 Stimmen


Die Stimmen je Adresse stimmen mit Tabelle 3 der Hinweise überein (Adresse 0: 2, 1: 2, 2 bis 5: je 1, 6: 2, 8: 1, 11: 1, 13: 1, 14: 1,
16: 1; alle anderen 0). Die drei größten Gebote (0, 1 und 6) decken nur die Häuser 0, 3, 6 ab (3 Stimmen), und auch mit den Geboten
mit je einer Stimme kommt man nie auf 5. Deshalb ist `(7, 12, 15)` stabil. `(0, 1, 2)` dagegen ist instabil: Das Programm findet
einen Gegenvorschlag, und mit `ja_stimmen` nachgerechnet bekommt er wirklich mindestens 5 von 9 Stimmen.

## Schritt 3: Alle Verteilungen durchprobieren (Siedlungs-Iteration)

Jetzt prüfen wir jede Verteilung `i < j < k`. `itertools.combinations(range(L), 3)` liefert sie alle. Wir sammeln die stabilen in
einer Liste und zählen, wie oft `gegenvorschlag` aufgerufen wurde. Das machen wir für die Beispiele 1 bis 3, bei den größeren würde es
zu lange dauern: Beispiel 4 hat schon 161 700 Verteilungen, Beispiel 7 sogar 40 495 000.

In [ ]:
from itertools import combinations


def stabile_alle(L, haeuser):
    stabile = []
    for V in combinations(range(L), 3):
        if ist_stabil(V, L, haeuser):
            stabile.append(V)
    return stabile


einfach = {}
for nr in (1, 2, 3):
    L, haeuser = lies(nr)
    einfach[nr] = stabile_alle(L, haeuser)
    anzahl_v = L * (L - 1) * (L - 2) // 6
    print('eisbuden%d.txt: L = %d, N = %d, %d Verteilungen geprüft, %d stabil' % (nr, L, len(haeuser), anzahl_v, len(einfach[nr])))
    print('   ', einfach[nr][:12], '...' if len(einfach[nr]) > 12 else '')

eisbuden1.txt: L = 20, N = 7, 1140 Verteilungen geprüft, 11 stabil
    [(2, 5, 14), (2, 6, 14), (2, 7, 14), (2, 8, 14), (2, 9, 14), (2, 10, 14), (2, 10, 15), (2, 11, 14), (2, 11, 15), (2, 12, 14), (2, 12, 15)] 
eisbuden2.txt: L = 50, N = 15, 19600 Verteilungen geprüft, 0 stabil
    [] 
eisbuden3.txt: L = 50, N = 16, 19600 Verteilungen geprüft, 2 stabil
    [(0, 17, 42), (1, 17, 42)] 


Das stimmt mit den Hinweisen überein: Bei `eisbuden1.txt` sind es 11 stabile Verteilungen, bei `eisbuden2.txt` keine und bei
`eisbuden3.txt` die zwei Verteilungen `(0, 17, 42)` und `(1, 17, 42)`.

## Schritt 4: Gegenvorschläge merken

Fast alle Verteilungen sind instabil, und oft schlägt derselbe Gegenvorschlag viele davon. Wir halten deshalb eine Liste `refs` von
Gegenvorschlägen bereit, jeweils als Liste der Abstände aller Häuser zur nächsten Bude. Für jede Verteilung `V` rechnen wir die Abstände
`d` aus und prüfen für jedes `ref` der Reihe nach: Stimmen mindestens `N // 2 + 1` Häuser dafür (`ref[h] < d[h]`)? Dann ist `V`
instabil, und wir sind fertig, ohne die Gebote zu bilden. Erst wenn kein `ref` schlägt, rufen wir das teure `gegenvorschlag` auf.
Findet es einen Gegenvorschlag, wandert er neu in `refs`; findet es keinen, ist `V` stabil.

Am Anfang füllen wir `refs` mit **lokalen Minima der Gesamtentfernung**: Wir starten mit zufälligen Buden und verschieben immer eine
Bude an die Adresse, die die Summe der Abstände aller Häuser zur nächsten Bude am kleinsten macht, bis sich nichts mehr ändert.
Solche Verteilungen gewinnen gegen die meisten anderen. (Zufallszahlen mit festem Startwert, damit alles reproduzierbar ist.)

In [ ]:
import random


def abstandstabelle(L, haeuser):
    return [[abstand(a, h, L) for h in haeuser] for a in range(L)]


def lokale_minima(D, versuche):
    """Abstieg der Gesamtentfernung; D[a][h] ist der Abstand von Adresse a zu Haus h."""
    zufall = random.Random(1)
    L, N = len(D), len(D[0])
    minima = set()
    for _ in range(versuche):
        V = zufall.sample(range(L), 3)
        while True:
            alt = tuple(V)
            for b in range(3):
                rest = [V[c] for c in range(3) if c != b]
                nah2 = [min(D[rest[0]][h], D[rest[1]][h]) for h in range(N)]
                beste, bester_wert = V[b], None
                for a in range(L):
                    wert = sum(min(D[a][h], nah2[h]) for h in range(N))
                    if bester_wert is None or wert < bester_wert:
                        beste, bester_wert = a, wert
                V[b] = beste
            if tuple(V) == alt:
                break
        minima.add(tuple(sorted(V)))
    return minima


def abstaende(V, D):
    return [min(x, y, z) for x, y, z in zip(D[V[0]], D[V[1]], D[V[2]])]


def stabile_mit_refs(L, haeuser, versuche=20):
    N = len(haeuser)
    mehrheit = N // 2 + 1
    D = abstandstabelle(L, haeuser)
    refs = [abstaende(V, D) for V in lokale_minima(D, versuche)]
    stabile, genau = [], 0
    for V in combinations(range(L), 3):
        d = abstaende(V, D)
        geschlagen = False
        for ref in refs:
            ja = 0
            for r, x in zip(ref, d):
                if r < x:
                    ja += 1
            if ja >= mehrheit:
                geschlagen = True
                break
        if geschlagen:
            continue
        genau += 1                         # kein bekannter Gegenvorschlag: jetzt gründlich prüfen
        W = gegenvorschlag(V, L, haeuser)
        if W is None:
            stabile.append(V)
        else:
            refs.append(abstaende(W, D))
    return stabile, genau, len(refs)


mit_refs = {}
for nr in (1, 2, 3, 4):
    L, haeuser = lies(nr)
    stabile, genau, nrefs = stabile_mit_refs(L, haeuser)
    mit_refs[nr] = stabile
    anzahl_v = L * (L - 1) * (L - 2) // 6
    print('eisbuden%d.txt: %d Verteilungen, nur %d gründlich geprüft (%d gemerkte Gegenvorschläge), %d stabil' %
          (nr, anzahl_v, genau, nrefs, len(stabile)))
for nr in (1, 2, 3):
    assert mit_refs[nr] == einfach[nr]

eisbuden1.txt: 1140 Verteilungen, nur 32 gründlich geprüft (22 gemerkte Gegenvorschläge), 11 stabil
eisbuden2.txt: 19600 Verteilungen, nur 35 gründlich geprüft (37 gemerkte Gegenvorschläge), 0 stabil
eisbuden3.txt: 19600 Verteilungen, nur 72 gründlich geprüft (72 gemerkte Gegenvorschläge), 2 stabil
eisbuden4.txt: 161700 Verteilungen, nur 70 gründlich geprüft (72 gemerkte Gegenvorschläge), 0 stabil


Die Ergebnisse sind dieselben wie bei der einfachen Siedlungs-Iteration (das prüft die letzte Zeile), aber es wird nur noch ein
winziger Bruchteil der Verteilungen gründlich geprüft (bei `eisbuden4.txt` 70 von 161 700). Neu ist `eisbuden4.txt`: Hier gibt es **keine** stabile Verteilung, wie in den Hinweisen.

## Schritt 5: Dasselbe mit numpy für die großen Beispiele

Bei `eisbuden5.txt` bis `eisbuden7.txt` sind es 2,4 Millionen bis 40 Millionen Verteilungen. Zwei Dinge machen das mit `numpy` machbar:

- `D` ist eine Tabelle (Zeile = Adresse, Spalte = Haus) mit den Abständen. Für **alle** Verteilungen mit kleinster Bude `i` auf einmal
  bekommt man die Abstände der Häuser als `minimum(PM[...], D[i])`; dabei ist `PM` die Tabelle der Minima je Adressenpaar `(j, k)`.
- Der Vergleich mit einem `ref` ist dann `count_nonzero(ref < d, axis=1) >= mehrheit` für alle Verteilungen gleichzeitig. Wir gehen die Referenzen
  der Reihe nach durch; nur die Überlebenden kommen zur nächsten Referenz.

Bei den Überlebenden, die gar keine Referenz schlägt, rufen wir wieder das gründliche `gegenvorschlag` auf (das neue `ref` bekommt
dann die Abstände des Gegenvorschlags). Der Abstieg arbeitet mit `numpy` ebenfalls schneller. Das ist dasselbe Verfahren wie in
Schritt 4. Die Zahl der gründlichen Prüfungen ist etwas größer, weil ein neuer Gegenvorschlag erst für die nächste Bude `i` gilt.

In [ ]:
import numpy as np


def abstandstabelle_np(L, haeuser):
    a = np.arange(L)[:, None]
    h = np.array(haeuser)[None, :]
    d = np.abs(a - h)
    return np.minimum(d, L - d).astype(np.int16)


def lokale_minima_np(D, versuche):
    zufall = random.Random(1)
    L = D.shape[0]
    minima = set()
    for _ in range(versuche):
        V = zufall.sample(range(L), 3)
        while True:
            alt = tuple(V)
            for b in range(3):
                nah2 = D[[V[c] for c in range(3) if c != b]].min(axis=0)
                V[b] = int(np.minimum(D, nah2[None, :]).sum(axis=1).argmin())
            if tuple(V) == alt:
                break
        minima.add(tuple(sorted(V)))
    return minima


def stabile_numpy(L, haeuser, versuche=20):
    N = len(haeuser)
    mehrheit = N // 2 + 1
    D = abstandstabelle_np(L, haeuser)
    refs = [D[list(V)].min(axis=0) for V in lokale_minima_np(D, versuche)]
    J, K = np.triu_indices(L, 1)                      # alle Paare j < k, sortiert nach j
    PM = np.minimum(D[J], D[K])                       # PM[p]: Abstände der Häuser zu den Buden j und k
    anfang = np.searchsorted(J, np.arange(L) + 1)     # ab hier ist j > i
    stabile, genau = [], 0
    for i in range(L - 2):
        Ji, Ki = J[anfang[i]:], K[anfang[i]:]
        d = np.minimum(PM[anfang[i]:], D[i][None, :])
        lebt = np.nonzero(np.count_nonzero(refs[0][None, :] < d, axis=1) < mehrheit)[0]
        for ref in refs[1:]:
            if len(lebt) == 0:
                break
            lebt = lebt[np.count_nonzero(ref[None, :] < d[lebt], axis=1) < mehrheit]
        for p in lebt:
            V = (i, int(Ji[p]), int(Ki[p]))
            genau += 1
            W = gegenvorschlag(V, L, haeuser)
            if W is None:
                stabile.append(V)
            else:
                refs.append(D[list(W)].min(axis=0))
    return stabile, genau, len(refs)


ergebnisse = {}
for nr in range(1, 8):
    L, haeuser = lies(nr)
    stabile, genau, nrefs = stabile_numpy(L, haeuser)
    ergebnisse[nr] = stabile
    anzahl_v = L * (L - 1) * (L - 2) // 6
    print('eisbuden%d.txt (L = %d, N = %d): %d Verteilungen, %d gründlich geprüft, %d stabil' %
          (nr, L, len(haeuser), anzahl_v, genau, len(stabile)))
for nr in (1, 2, 3, 4):
    assert ergebnisse[nr] == mit_refs[nr]

eisbuden1.txt (L = 20, N = 7): 1140 Verteilungen, 65 gründlich geprüft, 11 stabil
eisbuden2.txt (L = 50, N = 15): 19600 Verteilungen, 142 gründlich geprüft, 0 stabil
eisbuden3.txt (L = 50, N = 16): 19600 Verteilungen, 158 gründlich geprüft, 2 stabil
eisbuden4.txt (L = 100, N = 19): 161700 Verteilungen, 149 gründlich geprüft, 0 stabil
eisbuden5.txt (L = 247, N = 24): 2481115 Verteilungen, 490 gründlich geprüft, 6 stabil
eisbuden6.txt (L = 437, N = 36): 13813570 Verteilungen, 1120 gründlich geprüft, 0 stabil
eisbuden7.txt (L = 625, N = 40): 40495000 Verteilungen, 2708 gründlich geprüft, 16 stabil


## Die Ergebnisse

In [ ]:
for nr in range(1, 8):
    L, haeuser = lies(nr)
    stabile = ergebnisse[nr]
    print('eisbuden%d.txt (L = %d, N = %d): %s' % (nr, L, len(haeuser), '%d stabile Verteilungen' % len(stabile) if stabile else 'keine stabile Verteilung'))
    if stabile:
        zeilen = ', '.join('[%d, %d, %d]' % V for V in stabile[:6])
        print('    ', zeilen, '...' if len(stabile) > 6 else '')

eisbuden1.txt (L = 20, N = 7): 11 stabile Verteilungen
     [2, 5, 14], [2, 6, 14], [2, 7, 14], [2, 8, 14], [2, 9, 14], [2, 10, 14] ...
eisbuden2.txt (L = 50, N = 15): keine stabile Verteilung
eisbuden3.txt (L = 50, N = 16): 2 stabile Verteilungen
     [0, 17, 42], [1, 17, 42] 
eisbuden4.txt (L = 100, N = 19): keine stabile Verteilung
eisbuden5.txt (L = 247, N = 24): 6 stabile Verteilungen
     [83, 128, 231], [83, 129, 231], [83, 129, 232], [83, 130, 231], [83, 130, 232], [83, 130, 233] 
eisbuden6.txt (L = 437, N = 36): keine stabile Verteilung
eisbuden7.txt (L = 625, N = 40): 16 stabile Verteilungen
     [114, 285, 416], [114, 285, 417], [114, 285, 418], [114, 285, 419], [114, 285, 420], [114, 289, 416] ...


Das sind genau die Zahlen und Verteilungen der Lösungshinweise (Tabelle in Abschnitt 3.5): 11, 0, 2, 0, 6, 0 und 16 stabile
Verteilungen, also sind die Siedlungen 1, 3, 5 und 7 stabil und die Siedlungen 2, 4 und 6 nicht. Alle Listen stimmen Eintrag für Eintrag
(das prüft der Generator am Ende gegen die Tabelle der Hinweise). Jede gefundene Verteilung prüfen wir hier noch einmal mit dem
einfachen `ist_stabil` aus Schritt 2 nach, und wir suchen den besten Gegenvorschlag: Wie viele Häuser stimmen mindestens für ihn?

In [ ]:
def beste_stimmen(V, L, haeuser):
    """Größte Zahl von Häusern, die für irgendeine Verteilung W gegen V stimmen."""
    g = gebote(V, L, haeuser)
    beste = 0
    n = len(g)
    for x in range(n):
        sx, ax, mx = g[x]
        if 3 * sx <= beste:
            break
        for y in range(x + 1, n):
            sy, ay, my = g[y]
            if sx + 2 * sy <= beste:
                break
            mxy = mx | my
            for z in range(y + 1, n):
                sz, az, mz = g[z]
                if sx + sy + sz <= beste:
                    break
                beste = max(beste, (mxy | mz).bit_count())
    return beste


for nr in (1, 3, 5, 7):
    L, haeuser = lies(nr)
    N = len(haeuser)
    alle_stabil = all(ist_stabil(V, L, haeuser) for V in ergebnisse[nr])
    beste = [beste_stimmen(V, L, haeuser) for V in ergebnisse[nr]]
    print('eisbuden%d.txt: alle %d stabil: %s; größte Zahl von Ja-Stimmen gegen sie: %d bis %d (nötig wären %d von %d)' %
          (nr, len(beste), alle_stabil, min(beste), max(beste), N // 2 + 1, N))

eisbuden1.txt: alle 11 stabil: True; größte Zahl von Ja-Stimmen gegen sie: 3 bis 3 (nötig wären 4 von 7)
eisbuden3.txt: alle 2 stabil: True; größte Zahl von Ja-Stimmen gegen sie: 8 bis 8 (nötig wären 9 von 16)
eisbuden5.txt: alle 6 stabil: True; größte Zahl von Ja-Stimmen gegen sie: 12 bis 12 (nötig wären 13 von 24)
eisbuden7.txt: alle 16 stabil: True; größte Zahl von Ja-Stimmen gegen sie: 20 bis 20 (nötig wären 21 von 40)


Die gefundenen Verteilungen sind also wirklich stabil. Und sie sind alle nur „gerade so“ stabil: In allen vier Beispielen fehlt dem
besten Gegenvorschlag genau **eine** Stimme zur Mehrheit (3 statt 4, 8 statt 9, 12 statt 13, 20 statt 21). Bei 5 und 7 heißt das:
genau die Hälfte der Häuser stimmt dafür, aber „mehr Ja als Nein“ ist nicht erreicht. Das sagen auch die Hinweise.

**Vergleich mit den Hinweisen:** Die Hinweise brauchen für die optimierte Siedlungs-Iteration bei `eisbuden7.txt` 47,6 s. Zeiten geben
wir im Notebook nicht an (sie hängen vom Rechner ab); wichtiger ist die Zahl der gründlichen Prüfungen: nur 2708 von rund 40 Millionen
Verteilungen. Bei `eisbuden5.txt` sagt der Text der Hinweise „nur zwei“ stabile Verteilungen, die Tabelle zeigt wie wir sechs.
Der Text stimmt hier also nicht mit der eigenen Tabelle überein.

## Zum Schluss: Mit MiniZinc

Die Aufgabe lässt sich fast wörtlich als Modell schreiben: **Gesucht sind drei Buden `x`, sodass es keinen Gegenvorschlag gibt.** Du
kennst von den Arbeitsblättern zu MiniZinc 2D-Arrays, `sum … where`, `forall`, Variablen als Index und `.dzn`-Dateien. Neu ist hier nur,
dass wir die Bedingung für **alle** Gegenvorschläge `(a, b, c)` aufschreiben (`forall` über drei Indizes) und dass `nah[h]` von den
Variablen `x` abhängt. `bool2int` macht aus einer Aussage 0 oder 1; so zählt die `sum` die Häuser, die für `(a, b, c)` stimmen.

```minizinc
int: L;                                   % Umfang
int: N;                                   % Zahl der Häuser
set of int: ADR = 0..L-1;
set of int: HAUS = 1..N;
array[ADR, HAUS] of int: d;               % d[a, h]: Abstand von Adresse a zu Haus h
int: mehrheit = N div 2 + 1;

array[1..3] of var ADR: x;                % die gesuchten Buden
constraint x[1] < x[2] /\ x[2] < x[3];
array[HAUS] of var int: nah = [ min([d[x[1], h], d[x[2], h], d[x[3], h]]) | h in HAUS ];

% kein Gegenvorschlag (a, b, c) darf die Mehrheit bekommen
constraint forall(a, b, c in ADR where a < b /\ b < c)(
  sum(h in HAUS)(bool2int(min([d[a, h], d[b, h], d[c, h]]) < nah[h])) < mehrheit);

solve satisfy;
output ["\(x)\n"];
```

Die Abstandstabelle `d` kommt aus Python in die Datendatei (`als_dzn`). Mit `-a` (alle Lösungen) findet man alle stabilen Verteilungen.

In [ ]:
def als_dzn(L, haeuser):
    d = [abstand(a, h, L) for a in range(L) for h in haeuser]
    return 'L = %d;\nN = %d;\nd = array2d(0..%d, 1..%d, %s);' % (L, len(haeuser), L - 1, len(haeuser), d)


print(als_dzn(*lies(1))[:100], '...')

L = 20;
N = 7;
d = array2d(0..19, 1..7, [0, 2, 3, 8, 8, 6, 5, 1, 1, 2, 7, 9, 7, 6, 2, 0, 1, 6, 10, 8 ...


Ergebnis beim Erzeugen dieses Notebooks (Solver Chuffed, alle Lösungen, Zeitlimit etwa 2 Minuten; in MiniZinc über *Solver configuration*
gewählt). „gleich“ heißt: genau dieselben stabilen Verteilungen wie die Python-Lösung, „?“: Zeitlimit, keine Antwort. Die vierte Spalte
ist die Zahl der Bedingungen, die `forall` erzeugt.

| Beispiel | L | N | Bedingungen | Chuffed |
|----------|---|---|-------------|---------|
| `eisbuden1.txt` | 20 | 7 | 1140 | gleich (11 stabil) |
| `eisbuden2.txt` | 50 | 15 | 19600 | gleich (0 stabil) |
| `eisbuden3.txt` | 50 | 16 | 19600 | gleich (2 stabil) |
| `eisbuden4.txt` | 100 | 19 | 161700 | gleich (0 stabil) |
| `eisbuden5.txt` | 247 | 24 | 2481115 | ? |
| `eisbuden6.txt` | 437 | 36 | 13813570 | nicht versucht |
| `eisbuden7.txt` | 625 | 40 | 40495000 | nicht versucht |

Das Modell ist richtig und kurz, aber nur für kleine Siedlungen brauchbar: Die Zahl der Bedingungen wächst mit `L³`, und jede ist eine
Summe über alle Häuser. Schon bei `eisbuden4.txt` dauert es spürbar länger als unser Python-Programm, bei `eisbuden5.txt` kommt Chuffed im
Zeitlimit zu keinem Ergebnis, und `eisbuden6.txt` und `eisbuden7.txt` mit 14 und 40 Millionen Bedingungen haben wir gar nicht erst versucht. Unser Verfahren ist besser, weil es **nicht alle Gegenvorschläge für alle Verteilungen** aufschreibt, sondern sich die
wenigen merkt, die wirklich schlagen. Das ist ein schönes Beispiel dafür, dass man Wissen über die Aufgabe (hier die Gegenvorschläge)
besser nutzen kann als ein allgemeiner Löser.

## Für Neugierige

- **Laufzeit der einfachen Lösung:** Es gibt `O(L³)` Verteilungen, und das Gebote-System braucht pro Verteilung `O(L³ · N)`. Zusammen
  `O(L⁶ · N)` (Hinweise). Mit den gemerkten Gegenvorschlägen werden die meisten Verteilungen nach wenigen Vergleichen erledigt;
  die Zahl der gründlichen Prüfungen ist in allen Beispielen nur ein winziger Bruchteil.
- **Stabile Verteilungen liegen oft in der Nähe der lokalen Minima der Gesamtentfernung.** Die Hinweise nutzen deshalb den Abstieg
  auch, um stabile Verteilungen bei Beispiel 5 und 7 sehr schnell zu finden. Wir prüfen die lokalen Minima aus Schritt 4: Bei
  Beispiel 7 ist `(114, 285, 420)` schon selbst stabil, bei Beispiel 5 ist das Minimum `(83, 126, 230)` zwar instabil, liegt aber
  dicht bei den stabilen `(83, 128, 231)` bis `(83, 130, 233)`.

In [ ]:
for nr in (5, 7):
    L, haeuser = lies(nr)
    D = abstandstabelle_np(L, haeuser)
    for V in sorted(lokale_minima_np(D, 20)):
        print('eisbuden%d.txt, lokales Minimum der Gesamtentfernung %s: %s' % (nr, V, 'stabil' if ist_stabil(V, L, haeuser) else 'instabil'))

eisbuden5.txt, lokales Minimum der Gesamtentfernung (2, 97, 185): instabil
eisbuden5.txt, lokales Minimum der Gesamtentfernung (83, 126, 230): instabil
eisbuden7.txt, lokales Minimum der Gesamtentfernung (114, 285, 420): stabil


- **Gerade so stabil:** Bei den stabilen Verteilungen von 5 und 7 fehlt dem besten Gegenvorschlag genau eine Stimme. Entfernt man ein
  Haus oder zieht eines um, kann sich die Stabilität ändern. Das ist die Erweiterung, die die Hinweise vorschlagen.
- **Es gibt oft mehrere stabile Verteilungen nebeneinander** (`[114, 285, 416]` bis `[114, 285, 420]` bei Beispiel 7): Eine Bude
  darf häufig ein Stück wandern, ohne dass sich die Abstimmung ändert.